# 03 — Collaborative Filtering (SVD via scipy)
Learn user preferences from historical interactions using Matrix Factorization.

**No extra installs needed** — uses scipy and scikit-learn which are already installed.

How it works:
- Build a sparse user-item matrix from interaction weights
- Apply Truncated SVD to learn latent user and item factors
- Reconstruct predicted scores for unseen items

Run notebooks 01 and 02 first.

## 1. Load Data

In [ ]:
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
import pickle

train   = pd.read_csv('../data/processed/train.csv')
catalog = pd.read_csv('../data/processed/item_catalog.csv')

print(f'Train interactions: {len(train):,}')
print(f'Unique users:  {train["user_id"].nunique():,}')
print(f'Unique items:  {train["item_id"].nunique():,}')

## 2. Build User-Item Matrix

In [ ]:
# Aggregate duplicate user-item pairs by summing weights
agg = (
    train.groupby(['user_id', 'item_id'])['interaction_weight']
         .sum()
         .reset_index()
)

# Build integer index mappings
users = agg['user_id'].unique()
items = agg['item_id'].unique()

user_to_idx = {u: i for i, u in enumerate(users)}
item_to_idx = {it: i for i, it in enumerate(items)}
idx_to_user = {i: u for u, i in user_to_idx.items()}
idx_to_item = {i: it for it, i in item_to_idx.items()}

# Build sparse matrix
rows = agg['user_id'].map(user_to_idx).values
cols = agg['item_id'].map(item_to_idx).values
vals = agg['interaction_weight'].values.astype(np.float32)

user_item_matrix = sp.csr_matrix(
    (vals, (rows, cols)),
    shape=(len(users), len(items))
)

print(f'Matrix shape:    {user_item_matrix.shape}')
print(f'Non-zero values: {user_item_matrix.nnz:,}')
sparsity = 1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1])
print(f'Sparsity:        {sparsity:.4%}')

## 3. Train SVD Model
TruncatedSVD decomposes the matrix into user factors and item factors.
n_components = number of latent dimensions (like 'taste profiles').

In [ ]:
N_COMPONENTS = 50  # latent dimensions — increase for more accuracy, slower training

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

print(f'Training SVD with {N_COMPONENTS} components...')
user_factors = svd.fit_transform(user_item_matrix)  # shape: (n_users, n_components)
item_factors = svd.components_.T                    # shape: (n_items, n_components)

print(f'User factors shape: {user_factors.shape}')
print(f'Item factors shape: {item_factors.shape}')
print(f'Explained variance: {svd.explained_variance_ratio_.sum():.2%}')

## 4. Generate Recommendations

In [ ]:
# Build lookup of items each user has already seen
user_seen_items = (
    train.groupby('user_id')['item_id']
         .apply(set)
         .to_dict()
)

all_item_ids = catalog['item_id'].tolist()

def get_cf_recommendations(user_id, top_n=10):
    """
    Return top_n recommendations for a user using SVD dot-product scoring.
    Cold-start users (not seen during training) return empty DataFrame.
    """
    if user_id not in user_to_idx:
        print(f'User {user_id} not in training set (cold start).')
        return pd.DataFrame(columns=['item_id', 'item_title', 'cf_score'])

    u_idx    = user_to_idx[user_id]
    u_vector = user_factors[u_idx]  # (n_components,)

    # Score all items via dot product with item factors
    scores = item_factors @ u_vector  # (n_items,)

    # Exclude items the user has already seen
    seen      = user_seen_items.get(user_id, set())
    seen_idxs = [item_to_idx[i] for i in seen if i in item_to_idx]
    scores[seen_idxs] = -np.inf

    # Get top_n item indices
    top_idxs = np.argsort(scores)[::-1][:top_n]

    recs = pd.DataFrame({
        'item_id':  [idx_to_item[i] for i in top_idxs],
        'cf_score': scores[top_idxs].round(4)
    })
    recs = recs.merge(catalog[['item_id', 'item_title', 'avg_price']], on='item_id', how='left')
    return recs.reset_index(drop=True)


# Test it
sample_user = train['user_id'].value_counts().index[5]
print(f'SVD recommendations for user: {sample_user}')
display(get_cf_recommendations(sample_user, top_n=10))

## 5. Spot Check

In [ ]:
print(f"User {sample_user}'s top interactions:")
display(
    train[train['user_id'] == sample_user]
    .groupby('item_title')['interaction_weight'].sum()
    .nlargest(5).reset_index()
    .rename(columns={'interaction_weight': 'total_weight'})
)
print('\nTop 5 recommendations:')
display(get_cf_recommendations(sample_user, top_n=5)[['item_title', 'cf_score']])

## 6. Save Artifacts

In [ ]:
cf_data = {
    'svd':            svd,
    'user_factors':   user_factors,
    'item_factors':   item_factors,
    'user_to_idx':    user_to_idx,
    'item_to_idx':    item_to_idx,
    'idx_to_user':    idx_to_user,
    'idx_to_item':    idx_to_item,
    'user_seen_items': user_seen_items,
    'all_item_ids':   all_item_ids,
}

with open('../data/processed/cf_data.pkl', 'wb') as f:
    pickle.dump(cf_data, f)

print('Saved cf_data.pkl')
print('Run notebook 04 next.')